In [ ]:
import json
from torch.utils.data import Dataset
from transformers import RobertaTokenizer
import torch

class AIDetectorDataset(Dataset):
    def __init__(self, filepath, tokenizer, max_length=512):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                entry = json.loads(line.strip())
                self.samples.append({
                    'text': entry['text'],
                    'label': int(entry['label'])  # 0=human, 1=machine
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        encoding = self.tokenizer(
            item['text'],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(item['label'], dtype=torch.float)
        }

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import RobertaTokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

train_path = '/content/drive/My Drive/SubtaskA/subtaskA_train_monolingual.jsonl'
dev_path = '/content/drive/My Drive/SubtaskA/subtaskA_dev_monolingual.jsonl'

train_dataset = AIDetectorDataset(train_path, tokenizer)
dev_dataset = AIDetectorDataset(dev_path, tokenizer)

In [ ]:
import torch.nn as nn
from transformers import RobertaModel

class AIDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.classifier = nn.Sequential(
            nn.Linear(self.roberta.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)
        return logits

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32)

In [ ]:
import torch
from torch.optim import AdamW
from sklearn.metrics import accuracy_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AIDetector().to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.BCEWithLogitsLoss()

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).int()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"[Epoch {epoch+1}] Train loss: {total_loss:.4f}, Accuracy: {acc:.4f}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 7485/7485 [46:39<00:00,  2.67it/s]


[Epoch 1] Train loss: 206.9795, Accuracy: 0.9915


100%|██████████| 7485/7485 [46:18<00:00,  2.69it/s]


[Epoch 2] Train loss: 71.6666, Accuracy: 0.9970


100%|██████████| 7485/7485 [46:17<00:00,  2.69it/s]


[Epoch 3] Train loss: 54.3721, Accuracy: 0.9979


In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        logits = model(input_ids, attention_mask)
        preds = (torch.sigmoid(logits) > 0.5).int()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

val_acc = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Accuracy: 0.6986
